# Session 4 — nn.Module, layers, and the training loop

Companion to [../numpy_pytorch_schedule.md](../numpy_pytorch_schedule.md). Turns "I can do differentiable tensor ops" into "I can train a model." Three things: packaging parameters, the layers you'll use, and the loop you'll type a thousand times.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F

## nn.Module — packaging parameters

Register sub-modules/params in `__init__`, define `forward`. PyTorch tracks all `.parameters()` for the optimizer. Anything assigned as `self.x = nn.Module`/`nn.Parameter` is auto-registered.

In [3]:
class MLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()                       # always first
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

model = MLP(10, 32, 3)
print("params:", sum(p.numel() for p in model.parameters()))

params: 451


### What is `super().__init__()`?

Every `nn.Module` subclass starts its `__init__` with `super().__init__()`. Here's what it does and why it's mandatory:

- **`super()`** refers to the parent class — `nn.Module`. **`super().__init__()`** runs `nn.Module`'s own constructor, which creates the internal registries (`_parameters`, `_modules`, `_buffers`) that track everything you assign.
- **It must come first.** When you write `self.fc = nn.Linear(...)`, `nn.Module` intercepts the assignment (custom `__setattr__`) and files the layer into `_modules` so it appears in `.parameters()`, `state_dict`, `.to()`, etc. Those registries only exist *after* `super().__init__()` runs — so assigning a layer before it raises `cannot assign module before Module.__init__() call`.

**Rule:** call `super().__init__()` as the first line of every `nn.Module` subclass's `__init__`, before creating any layers or parameters. Run the cell below to see both the success and the error:

In [5]:
# WITH super().__init__(): registration machinery is set up first
class Good(nn.Module):
    def __init__(self):
        super().__init__()                 # creates nn.Module's _parameters/_modules/_buffers
        self.fc = nn.Linear(4, 4)          # __setattr__ files this into _modules -> registered

print("Good ->", [n for n, _ in Good().named_parameters()])   # ['fc.weight', 'fc.bias']

# WITHOUT it: assigning a layer fails because the registries don't exist yet
class Bad(nn.Module):
    def __init__(self):
        self.fc = nn.Linear(4, 4)          # no super().__init__() ran first

try:
    Bad()
except AttributeError as e:
    print("Bad ->", e)                     # cannot assign module before Module.__init__() call

Good -> ['fc.weight', 'fc.bias']
Bad -> cannot assign module before Module.__init__() call


## What `nn.Module` provides for you

`nn.Module` is the base class for models *and* individual layers. Its job is **bookkeeping** — it tracks every parameter and piece of state so you don't manage them by hand:

- **Automatic parameter registration.** Assign an `nn.Parameter` or a sub-`nn.Module` as an attribute and it's auto-registered; `model.parameters()` returns *every* param, **recursing the whole tree** — that's what you hand the optimizer.
- **Buffers** (`register_buffer`): non-learnable state that must travel with the model (causal mask, RoPE table, BatchNorm running stats). In `state_dict` and moved by `.to()`, but **not** in `.parameters()` (the optimizer never updates it).
- **`state_dict` / `load_state_dict`**: save/load checkpoints (all params + buffers).
- **Recursive device/dtype movement**: `.to("cuda")`, `.half()` move everything at once.
- **`train()` / `eval()`**: flips `self.training` recursively (Dropout/BatchNorm change behavior). Pair `eval()` with `torch.no_grad()`.
- **`__call__` → `forward`**: `model(x)` runs `forward(x)` **plus** hooks and autograd bookkeeping — which is why you call `model(x)`, never `model.forward(x)`.
- **`apply(fn)`**: recursively apply a function to every submodule — the standard weight-init idiom (`self.apply(self._init)` in Session 5).
- **Hooks** (`register_forward_hook`, …) for instrumentation, and a readable `print(model)` tree.

One line: **`nn.Module` turns a pile of tensors into a managed tree** — every parameter findable, all state saveable, everything movable to a device and toggleable train/eval, recursively. Doing this by hand with dicts of tensors is the tedium it removes.

## Class-based vs. functional — which to use

Not either/or. The rule:

> **Anything with learnable parameters or state → a class (`nn.Module`). Anything stateless → a function (`torch.nn.functional`, `F.*`). Inside `forward`, you mix the two.**

Parameters *need* the registration/saving/device machinery above, and only `nn.Module` provides it — so your **model is always a class**. Stateless ops (activations, softmax, the loss) have nothing to track, so they're plain `F.*` calls inline:

```python
class Block(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.ln  = nn.LayerNorm(D)     # has params -> Module
        self.mlp = nn.Linear(D, D)     # has params -> Module
    def forward(self, x):
        return x + self.mlp(F.gelu(self.ln(x)))   # F.gelu stateless -> function
```

Nuances:
- **`nn.ReLU` vs `F.relu`** (and `nn.Softmax` vs `F.softmax`): neither has params, so it's stylistic — use `nn.*` when building with `nn.Sequential`, `F.*` inline in a custom `forward` (less boilerplate).
- **`nn.Sequential`** is the middle ground: applies submodules **in order**, no branching. Great for a plain MLP; but it **can't** express residuals, multiple inputs/outputs, or conditionals — the moment you need `x = x + sublayer(x)`, write an explicit `forward` class (that's why Session 5's block is a class).
- **Purely functional models** aren't idiomatic in mainstream PyTorch (that's the JAX/Flax world); `torch.func` exists for `vmap`/`grad` transforms but is niche. For normal model building, class-based `nn.Module` is the standard.

## Best-practices checklist

1. Subclass `nn.Module`; call `super().__init__()` **first**.
2. Create all layers/params in `__init__`; compute in `forward`. **Never create layers inside `forward`** — they'd be re-created each call and unregistered.
3. Lists/dicts of submodules → **`nn.ModuleList` / `nn.ModuleDict`**, never a plain `list`/`dict` (gotcha below).
4. **`register_buffer`** for non-learnable tensors that must move/save with the model (masks, fixed positional tables).
5. Call **`model(x)`**, not `model.forward(x)`.
6. `F.*` for stateless ops; `nn.*` modules for parameterized layers; `nn.Sequential` only for simple in-order chains.
7. Init with **`self.apply(init_fn)`**.
8. Toggle **`train()`/`eval()`**; wrap inference in **`torch.no_grad()`**.
9. Parameterize dims in `__init__`; keep `forward` shape-clean.
10. Keep the model definition separate from the training loop.

## The #1 gotcha: plain list vs. `nn.ModuleList`

A plain Python list of submodules does **not** register them — their parameters vanish from `.parameters()` and the optimizer silently never trains them. Run this and compare the counts:

In [ ]:
class WithPlainList(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 4) for _ in range(3)]              # plain list -> NOT registered

class WithModuleList(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList(nn.Linear(4, 4) for _ in range(3)) # registered

print("plain list -> params:", sum(p.numel() for p in WithPlainList().parameters()))   # 0  (!!)
print("ModuleList  -> params:", sum(p.numel() for p in WithModuleList().parameters()))  # 60

# buffers: in state_dict + moved by .to(), but NOT in parameters()
class WithBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(4, 4)
        self.register_buffer("mask", torch.ones(4, 4))
m = WithBuffer()
print("state_dict keys:", list(m.state_dict().keys()))   # includes 'mask' + 'lin.weight'/'lin.bias'
print("buffer in parameters()?", any(t is m.mask for t in m.parameters()))  # False

## The layers you actually need

| Layer | What it does |
|---|---|
| `nn.Linear(in, out)` | `y = xW + b` (`bias=False` to drop bias) |
| `nn.Embedding(V, D)` | token-id → `D`-vector lookup |
| `nn.LayerNorm(D)` | per-token normalize + learnable gain/bias (Session 1's op) |
| `F.cross_entropy(logits, targets)` | softmax + NLL in one (stable) |
| `F.gelu` / `F.silu` | activations |
| `F.scaled_dot_product_attention` | fused attention (Session 5) |

`F.cross_entropy` expects **raw logits** `(N, V)` and integer `targets` `(N,)` — it log-softmaxes internally, so never softmax before it.

In [4]:
logits = torch.randn(5, 3)           # (N, V) raw logits
targets = torch.randint(0, 3, (5,))  # (N,) integer classes
print("loss:", F.cross_entropy(logits, targets).item())
# LayerNorm reproduces Session 1's standardize (+ learnable gain/bias):
ln = nn.LayerNorm(4)
print("LN out row mean ~0:", ln(torch.randn(2, 4)).mean(dim=-1).detach())

loss: 1.9096310138702393
LN out row mean ~0: tensor([1.4901e-08, 2.9802e-08])


## The training loop — memorize this rhythm

Every run is this five-line core. The order of 3–4–5 is not negotiable, and **`zero_grad` before `backward`** is the one people forget (recall: `.grad` accumulates).

```
logits = model(x)                    # 1. forward
loss = F.cross_entropy(logits, y)    # 2. loss (scalar)
optimizer.zero_grad()                # 3. clear last step's grads
loss.backward()                      # 4. backprop -> fills .grad
optimizer.step()                     # 5. update params
```

`loss` here is a **scalar tensor** (0-dim, shape `()`) that carries the autograd graph — that's why `loss.backward()` needs no arguments (it seeds `d(loss)/d(loss)=1`) and why we print `loss.item()` (which pulls the plain float off the tensor). `AdamW` is the Part-2.4 optimizer — hand it `model.parameters()` and it owns the update + per-param state.

## Data — just enough

`Dataset` + `DataLoader` batch and shuffle. `TensorDataset` is plenty for learning.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
X = torch.randn(128, 10); Y = torch.randint(0, 3, (128,))
loader = DataLoader(TensorDataset(X, Y), batch_size=32, shuffle=True)
xb, yb = next(iter(loader))
print("batch:", xb.shape, yb.shape)

## Self-check

1. Why `zero_grad()` *before* `backward()`? What breaks if omitted?
2. `F.cross_entropy` — probabilities or raw logits, and why?
3. How does the optimizer know which tensors to update, and where do its gradients come from?

**Answers.** (1) `.grad` accumulates; zeroing first ensures `.grad` holds only this step's gradient before `step()`. Omit it and old gradients pile up → training destabilizes. (2) Raw **logits** — it applies `log_softmax` internally (stable); softmaxing first double-applies it. (3) You gave it `model.parameters()`; `backward()` fills each param's `.grad`; `step()` reads those and applies AdamW in place.

## Exercise — train an MLP on a learnable rule

`X = torch.randn(512, 10)`, `Y = X[:, :3].argmax(dim=1)` (label = which of the first 3 features is largest). Write the loop over ~300 steps with `AdamW`; loss should fall sharply from ~`ln(3)≈1.1` toward 0. Then `.to(device)` and confirm it still runs.

In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

torch.manual_seed(0)
X = torch.randn(512, 10, device=device)
Y = X[:, :3].argmax(dim=1)                       # learnable rule
model = MLP(10, 32, 3).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(300):
    logits = model(X)
    loss = F.cross_entropy(logits, Y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(step, round(loss.item(), 3))

acc = (model(X).argmax(1) == Y).float().mean().item()
print("final train accuracy:", round(acc, 3))    # ~1.0

A *learnable* rule is the honest test: loss → ~0, accuracy → ~100%. (Random labels would barely move at this size — real learning needs signal in the data.) Session 5 swaps this `MLP` for a Transformer block; the loop is unchanged.